# Augmentation فیزیکی کامل با Point Group Symmetry + بازآموزی معماری v4

## پیشینه
- **Notebook 15 (v4)**: بهترین نتیجه‌ی تاکنون — MAE = 0.909 THz (پیش‌بینی فقط بلوک سلول
  مرجع IFC، بقیه از ground truth قرض گرفته شده برای ارزیابی)
- **Notebooks 16-18**: تلاش برای پیش‌بینی کامل ماتریس IFC با factorization — همه بدتر از v4
  (نتیجه‌گیری: مشکل کیفیت سیگنال نبود، خود تجزیه‌ی پیش‌بینی به اجزای مستقل مشکل‌ساز بود)
- **Notebook 19 (v1-v3)**: کشف ۲۴ عملیات تقارن نقطه‌ای (`P6₃/mmc`) و تأیید کامل فرمول
  transform فیزیکی روی یک ماده‌ی نمونه — هر ۲۴ عملیات با خطای فرکانس دقیقاً صفر تأیید شدند

## هدف این نوت‌بوک
اعمال augmentation فیزیکی (با فرمول تست‌شده‌ی notebook 19) روی **تمام ۳۵۸ ماده** برای
گسترش داده‌ی آموزشی، سپس بازآموزی معماری v4 (notebook 15، بدون تغییر معماری) روی این
داده‌ی گسترش‌یافته — برای دیدن این‌که آیا augmentation به‌تنهایی (بدون تغییر معماری)
می‌تواند MAE را زیر 0.909 ببرد.

## نکات طراحی مهم
1. **جلوگیری از Data Leakage**: تقسیم Train/Val/Test ابتدا روی ۳۵۸ ماده‌ی اصلی انجام
   می‌شود (دقیقاً مثل notebook 15، `seed=42`)، و augmentation **فقط روی نمونه‌های Train**
   اعمال می‌شود — Val و Test دست‌نخورده باقی می‌مانند تا ارزیابی واقعی و بدون تورش باشد.
2. **فیلتر عملیات تکراری**: برای هر ماده، اگر دو عملیات تقارن یک ساختار transform‌شده‌ی
   کاملاً یکسان (تا تلورانس عددی) تولید کنند، فقط یکی نگه داشته می‌شود تا داده‌ی بی‌فایده
   اضافه نشود.
3. فقط محور سلول مرجع IFC نیاز است (نه کل ماتریس سوپرسل)، چون معماری v4 فقط همان بلوک را
   پیش‌بینی می‌کند -- اما برای محاسبه‌ی صحیح transform باید کل محاسبات (هر دو محور) انجام
   شود و در پایان فقط بلوک مرجع برای آموزش نگه داشته می‌شود.

In [ ]:
!pip install -q phonopy torch_geometric pymatgen torch-cluster torch-sparse torch-scatter torch-spline-conv spglib -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("Installed")

In [1]:
!pip install torch_geometric 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.5 MB/s eta 0:00:00


In [2]:
!pip install phonopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.3/787.3 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00


In [3]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from sklearn.metrics import mean_absolute_error
from tqdm.notebook import tqdm

from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS
import spglib

device = torch.device('cpu')
print(f"Device: {device}")

Device: cpu


## مسیرهای داده

In [4]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = 'OK' if os.path.exists(path) else 'MISSING'
    print(f"[{exists}]  {name}  ->  {path}")

[OK]  FC_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C
[OK]  POSCAR_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar
[OK]  BANDS_DIR  ->  /kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C
[OK]  ELASTIC_FILE  ->  /kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json
[OK]  FEATURE_CSV  ->  /kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv


## کشف خودکار Supercell صحیح (مستقل برای هر ماده) — بدون تغییر نسبت به نوت‌بوک‌های قبلی

In [5]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    masses = [p['mass'] for p in data['points']]
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc

    return {
        'phonon': phonon,
        'symbols': symbols,
        'frac_coords': frac_coords,
        'lattice': lattice,
        'masses': masses,
        'supercell_dim': dim,
        'dim_match_error': err,
        'force_constants': fc,
        'all_q_points': [p['q-position'] for p in data['phonon']],
        'all_real_freqs': np.array([sorted([b['frequency'] for b in p['band']]) for p in data['phonon']]),
    }

print("Supercell discovery functions ready")

Supercell discovery functions ready


## توابع Augmentation (تأیید‌شده در notebook 19، v3 — خطای فرکانس صفر روی هر ۲۴ عملیات)

In [6]:
def find_supercell_permutation(supercell, R_cart, t_cart, symprec=1e-3):
    """
    Apply the cartesian symmetry operation (R_cart, t_cart) to every supercell atom
    position and find the matching original supercell atom (same element, same
    position modulo the supercell lattice). Returns a permutation array of length
    n_supercell, or None if no valid bijection exists.
    """
    sc_cart = supercell.positions
    sc_symbols = supercell.symbols
    sc_lattice = supercell.cell
    inv_lattice = np.linalg.inv(sc_lattice)

    n = len(sc_cart)
    transformed = sc_cart @ R_cart.T + t_cart

    perm = np.full(n, -1, dtype=int)
    for i in range(n):
        best_j, best_dist = -1, np.inf
        for j in range(n):
            if sc_symbols[j] != sc_symbols[i]:
                continue
            diff_cart = transformed[i] - sc_cart[j]
            diff_frac = diff_cart @ inv_lattice
            diff_frac = diff_frac - np.round(diff_frac)
            diff_cart_wrapped = diff_frac @ sc_lattice
            dist = np.linalg.norm(diff_cart_wrapped)
            if dist < best_dist:
                best_dist = dist
                best_j = j
        if best_dist > symprec:
            return None
        perm[i] = best_j

    if len(set(perm.tolist())) != n:
        return None
    return perm


def cartesian_rotation(R_frac, lattice):
    L = lattice
    return L.T @ R_frac @ np.linalg.inv(L.T)


def cartesian_translation(t_frac, lattice):
    return t_frac @ lattice


def supercell_index_to_unitcell_axis(sc_idx, s2u_map, unitcell_sc_indices):
    rep = s2u_map[sc_idx]
    matches = np.where(unitcell_sc_indices == rep)[0]
    if len(matches) == 0:
        return None
    return matches[0]


def transform_ifc_full(fc, perm_unitcell_into_sc_indices, perm_supercell, R_cart):
    """Transform the full IFC tensor (n_atoms, n_supercell, 3, 3) under a symmetry operation."""
    fc_permuted = fc[perm_unitcell_into_sc_indices][:, perm_supercell]
    fc_new = np.einsum('ab,nscd,db->nsac', R_cart, fc_permuted, R_cart)
    return fc_new


def generate_symmetry_augmented_samples(phonon, fc, lattice, frac_coords, symbols, n_atoms,
                                          dedup_decimals=4, symprec=1e-3):
    """
    Apply all point-group symmetry operations to (lattice, frac_coords, fc) and return a
    list of valid, deduplicated augmented samples:
    [{'frac_coords': ..., 'force_constants_unitcell_block': (n_atoms, n_atoms, 3, 3), 'op_index': i}, ...]
    Only the reference-atom block (first n_atoms columns of the supercell axis, matching
    the unit-cell representative order) is kept, since that's what the v4 architecture uses.
    Deduplicates augmented structures that produce identical (rounded) fractional
    coordinates to an already-kept sample.
    """
    cell_tuple = (lattice, frac_coords, [ord(s[0]) for s in symbols])  # numbers placeholder, real symbols used separately
    # use proper atomic numbers via PhonopyAtoms for spglib compatibility
    unitcell_obj = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)
    cell_tuple = (unitcell_obj.cell, unitcell_obj.scaled_positions, unitcell_obj.numbers)
    dataset = spglib.get_symmetry_dataset(cell_tuple, symprec=symprec)
    if dataset is None:
        return []

    s2u_map = phonon.supercell.s2u_map
    unitcell_sc_indices = np.unique(s2u_map)
    if len(unitcell_sc_indices) != n_atoms:
        return []
    unitcell_axis_to_sc_idx = unitcell_sc_indices

    samples = []
    seen_keys = set()

    for op_idx in range(len(dataset.rotations)):
        R = dataset.rotations[op_idx]
        t = dataset.translations[op_idx]
        R_cart = cartesian_rotation(R, lattice)
        t_cart = cartesian_translation(t, lattice)

        full_perm = find_supercell_permutation(phonon.supercell, R_cart, t_cart, symprec=symprec)
        if full_perm is None:
            continue

        perm_unitcell_axis = np.full(n_atoms, -1, dtype=int)
        valid = True
        for axis_i in range(n_atoms):
            sc_idx = unitcell_axis_to_sc_idx[axis_i]
            target_sc_idx = full_perm[sc_idx]
            target_axis = supercell_index_to_unitcell_axis(target_sc_idx, s2u_map, unitcell_sc_indices)
            if target_axis is None:
                valid = False
                break
            perm_unitcell_axis[axis_i] = target_axis
        if not valid:
            continue

        # transformed fractional coordinates of the unit-cell atoms (for graph features)
        new_frac = (frac_coords @ R.T + t) % 1.0

        dedup_key = tuple(np.round(new_frac[np.lexsort(new_frac.T)], decimals=dedup_decimals).flatten())
        if dedup_key in seen_keys:
            continue
        seen_keys.add(dedup_key)

        fc_full_transformed = transform_ifc_full(fc, perm_unitcell_axis, full_perm, R_cart)
        # keep only the reference-block columns: the n_atoms supercell columns that are the
        # unit-cell representatives themselves (matching how notebook 15's v4 dataset was built)
        fc_unitcell_block = fc_full_transformed[:, unitcell_sc_indices, :, :]  # (n_atoms, n_atoms, 3, 3)

        samples.append({
            'frac_coords': new_frac.astype(np.float32),
            'force_constants_unitcell_block': fc_unitcell_block.astype(np.float32),
            'op_index': op_idx,
        })

    return samples

print("Augmentation functions ready")

Augmentation functions ready


## ساخت دیتاست پایه (۳۵۸ ماده، بدون augmentation) — برای کشف Supercell و تقسیم Train/Val/Test

In [7]:
POSCAR_DIR_PATH = Path(POSCAR_DIR)
FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}
poscar_files = {f.stem: f for f in POSCAR_DIR_PATH.glob('*.psc')}

common = sorted(set(fc_files) & set(band_yaml_files) & set(poscar_files))
print(f"Number of common materials: {len(common)}")

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

Number of common materials: 358


In [8]:
raw_samples = []
failed_phonopy = []
failed_other = []

N_ATOMS_FIXED = 8
N_SUPERCELL_FIXED = 72

for formula in tqdm(common, desc="Loading materials with Phonopy"):
    try:
        with open(band_yaml_files[formula]) as f:
            quick_check = yaml.safe_load(f)
        if quick_check['natom'] != N_ATOMS_FIXED:
            continue

        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            failed_phonopy.append(formula)
            continue

        if result['force_constants'].shape != (N_ATOMS_FIXED, N_SUPERCELL_FIXED, 3, 3):
            continue

        # reference block for the ORIGINAL (non-augmented) sample, matching notebook 15's v4 setup
        s2u_map = result['phonon'].supercell.s2u_map
        unitcell_sc_indices = np.unique(s2u_map)
        if len(unitcell_sc_indices) != N_ATOMS_FIXED:
            continue
        fc_unitcell_block = result['force_constants'][:, unitcell_sc_indices, :, :]

        positions_cart = result['frac_coords'] @ result['lattice']

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           N_ATOMS_FIXED,
            'lattice':           result['lattice'].astype(np.float32),
            'frac_coords':       result['frac_coords'].astype(np.float32),
            'positions':         positions_cart.astype(np.float32),
            'atom_elements':     result['symbols'],
            'force_constants_unitcell_block': fc_unitcell_block.astype(np.float32),  # (8,8,3,3)
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          result['all_real_freqs'].astype(np.float32),
            'phonon_obj':        result['phonon'],
            'fc_full':           result['force_constants'].astype(np.float32),  # kept for augmentation step
        })
    except Exception as e:
        failed_other.append((formula, str(e)))

print(f"\nSucceeded: {len(raw_samples)}")
print(f"Failed (supercell discovery): {len(failed_phonopy)}")
print(f"Failed (other errors): {len(failed_other)}")

Loading materials with Phonopy:   0%|          | 0/358 [00:00<?, ?it/s]

/tmp/ipykernel_58/3492352745.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
/usr/local/lib/python3.12/dist-packages/phonopy/api_phonopy.py:278: UserWarning: Warning: Point group symmetries of supercell and primitivecell are different.
  self._search_primitive_symmetry()



Succeeded: 358
Failed (supercell discovery): 0
Failed (other errors): 0


## تقسیم Train / Val / Test (seed=42) — مشابه notebook 15، قبل از augmentation

In [9]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
print("Augmentation will be applied ONLY to the train split to avoid leakage into val/test.")

Train: 286 | Val: 35 | Test: 37
Augmentation will be applied ONLY to the train split to avoid leakage into val/test.


## اعمال Augmentation فقط روی Train (با فرمول تأیید‌شده‌ی notebook 19)

برای هر ماده‌ی train، تا ۲۴ نمونه‌ی جدید (بعد از dedup) ساخته می‌شود. هر نمونه شامل
fractional coordinates transform‌شده و بلوک IFC مرجع transform‌شده است.

In [10]:
augmented_train_samples = []
augmentation_stats = Counter()

for i in tqdm(train_idx, desc="Generating symmetry-augmented samples for train split"):
    s = raw_samples[i]
    try:
        aug_list = generate_symmetry_augmented_samples(
            s['phonon_obj'], s['fc_full'], s['lattice'], s['frac_coords'],
            s['atom_elements'], s['n_atoms'])
    except Exception as e:
        aug_list = []

    augmentation_stats[len(aug_list)] += 1

    for aug in aug_list:
        positions_cart_aug = aug['frac_coords'] @ s['lattice']
        augmented_train_samples.append({
            'formula':           s['formula'] + f"_aug{aug['op_index']}",
            'n_atoms':           s['n_atoms'],
            'lattice':           s['lattice'],
            'positions':         positions_cart_aug.astype(np.float32),
            'atom_elements':     s['atom_elements'],
            'force_constants_unitcell_block': aug['force_constants_unitcell_block'],
            'elastic_constants': s['elastic_constants'],
        })

print(f"\nOriginal train samples: {len(train_idx)}")
print(f"Augmented samples generated: {len(augmented_train_samples)}")
print(f"Total effective train samples: {len(train_idx) + len(augmented_train_samples)}")
print(f"\nDistribution of augmented-samples-per-material (including the identity op):")
for k, v in sorted(augmentation_stats.items()):
    print(f"   {k} valid ops: {v} materials")

Generating symmetry-augmented samples for train split:   0%|          | 0/286 [00:00<?, ?it/s]


Original train samples: 286
Augmented samples generated: 345
Total effective train samples: 631

Distribution of augmented-samples-per-material (including the identity op):
   1 valid ops: 227 materials
   2 valid ops: 59 materials


## ساخت گراف (Bond + Atom) — بدون تغییر نسبت به notebook 15

In [11]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"Atomic features: {N_ATOM_FEATURES} -> {feature_cols}")


def structure_to_bond_graph(positions):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)


def structure_to_atom_graph(atom_elements, positions):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

print("Graph construction functions ready")

Atomic features: 7 -> ['atomic_weight', 'atomic_radius', 'atomic_radius_rahm', 'covalent_radius_cordero', 'vdw_radius', 'atomic_volume', 'lattice_constant']
Graph construction functions ready


In [12]:
# Build graphs for: original samples (all splits) + augmented samples (train only)
bond_graphs_all, atom_graphs_all, ifc_targets_all = [], [], []
sample_id_map = {}  # maps a logical "train pool index" to position in the arrays

for i, s in enumerate(tqdm(raw_samples, desc="Building graphs for original samples")):
    bond_graphs_all.append(structure_to_bond_graph(s['positions']))
    atom_graphs_all.append(structure_to_atom_graph(s['atom_elements'], s['positions']))
    ifc_targets_all.append(s['force_constants_unitcell_block'])

n_original = len(raw_samples)

for s in tqdm(augmented_train_samples, desc="Building graphs for augmented samples"):
    bond_graphs_all.append(structure_to_bond_graph(s['positions']))
    atom_graphs_all.append(structure_to_atom_graph(s['atom_elements'], s['positions']))
    ifc_targets_all.append(s['force_constants_unitcell_block'])

# final index sets: train now includes original train indices PLUS all augmented sample indices
augmented_indices = list(range(n_original, n_original + len(augmented_train_samples)))
train_idx_augmented = np.concatenate([train_idx, np.array(augmented_indices, dtype=int)])

print(f"Total graphs built: {len(bond_graphs_all)}")
print(f"Effective train set size (original + augmented): {len(train_idx_augmented)}")
print(f"Val set size (unchanged): {len(val_idx)}")
print(f"Test set size (unchanged): {len(test_idx)}")

Building graphs for original samples:   0%|          | 0/358 [00:00<?, ?it/s]

Building graphs for augmented samples:   0%|          | 0/345 [00:00<?, ?it/s]

Total graphs built: 703
Effective train set size (original + augmented): 631
Val set size (unchanged): 35
Test set size (unchanged): 37


## معماری مدل: v4 (Dual Graph GNN، فقط بلوک سلول مرجع) — بدون تغییر نسبت به notebook 15

In [13]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphReferenceBlockIFCNet(nn.Module):
    """v4 architecture: predicts only the (n_atoms, n_atoms, 3, 3) reference block of the IFC tensor."""
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1, n_atoms_unitcell=8):
        super().__init__()
        self.n_atoms_unitcell = n_atoms_unitcell
        hidden = 128 if n_atom_features <= 10 else 96

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(5)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(5)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(2)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        combined_dim = 2*hidden + hidden + hidden + 2*hidden + hidden + hidden + hidden // 4
        ifc_out_dim = n_atoms_unitcell * n_atoms_unitcell * 3 * 3  # 8*8*3*3 = 576

        self.ifc_head = nn.Sequential(
            nn.Linear(combined_dim, 1024), nn.LayerNorm(1024), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(1024, 512), nn.LayerNorm(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, ifc_out_dim)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)

        ifc_flat = self.ifc_head(combined)
        batch_size = ifc_flat.shape[0]
        ifc_pred = ifc_flat.view(batch_size, self.n_atoms_unitcell, self.n_atoms_unitcell, 3, 3)
        return ifc_pred

print("DualGraphReferenceBlockIFCNet (v4 architecture) defined")

DualGraphReferenceBlockIFCNet (v4 architecture) defined


## Physics-Informed Loss — ASR (مشابه notebook 15)

In [14]:
def acoustic_sum_rule_loss(ifc_pred):
    """ASR over the reference block (batch, n_atoms, n_atoms, 3, 3): sum over axis=2 should be zero."""
    asr = torch.sum(ifc_pred, dim=2)
    return torch.mean(asr ** 2)


LAMBDA_ASR = 0.1


def compute_batch_loss(model, bond_data, atom_data, ifc_target_batch, device):
    ifc_pred = model(bond_data, atom_data)
    ifc_true = torch.tensor(np.stack(ifc_target_batch), dtype=torch.float32, device=device)

    mse_loss = F.mse_loss(ifc_pred, ifc_true)
    asr_loss = acoustic_sum_rule_loss(ifc_pred)

    total_loss = mse_loss + LAMBDA_ASR * asr_loss
    return total_loss, mse_loss.item(), asr_loss.item(), ifc_pred

print(f"Combined loss ready: lambda_asr={LAMBDA_ASR}")

Combined loss ready: lambda_asr=0.1


## حلقه‌ی آموزش

In [15]:
def make_batches(indices, batch_size, shuffle=False):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = np.random.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch(model, indices, optimizer=None, batch_size=16, shuffle=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_mse, total_asr = 0., 0., 0.
    n_batches = 0

    for batch_idx in make_batches(indices, batch_size, shuffle):
        bond_list = [bond_graphs_all[i] for i in batch_idx]
        atom_list = [atom_graphs_all[i] for i in batch_idx]
        ifc_list = [ifc_targets_all[i] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            loss, mse_v, asr_v, _ = compute_batch_loss(model, bond_batch, atom_batch, ifc_list, device)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        total_mse += mse_v
        total_asr += asr_v
        n_batches += 1

    n_batches = max(n_batches, 1)
    return total_loss/n_batches, total_mse/n_batches, total_asr/n_batches

print("run_epoch ready")

run_epoch ready


In [16]:
model = DualGraphReferenceBlockIFCNet(n_bond_features=6, n_atom_features=N_ATOM_FEATURES,
                                        edge_dim=1, n_atoms_unitcell=8).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

EPOCHS = 500
PATIENCE = 100
BATCH_SIZE_TRAIN = 16  # larger batch since the augmented train set is much bigger

best_val_loss = float('inf')
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_mse, train_asr = run_epoch(
        model, train_idx_augmented, optimizer=optimizer, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loss, val_mse, val_asr = run_epoch(
        model, val_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, '/kaggle/working/best_v4_augmented_model.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
              f"(MSE={val_mse:.4f}, ASR={val_asr:.4f}) Best: {best_val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

Epoch    0 | Train: 3.6958 | Val: 0.8791 (MSE=0.8170, ASR=0.6205) Best: 0.8791
Epoch   10 | Train: 0.7412 | Val: 0.5490 (MSE=0.5033, ASR=0.4574) Best: 0.5490
Epoch   20 | Train: 0.6262 | Val: 0.5230 (MSE=0.4727, ASR=0.5036) Best: 0.4893
Epoch   30 | Train: 0.5421 | Val: 0.4701 (MSE=0.4281, ASR=0.4200) Best: 0.4536
Epoch   40 | Train: 0.5163 | Val: 0.4271 (MSE=0.3810, ASR=0.4608) Best: 0.4271
Epoch   50 | Train: 0.4894 | Val: 0.4379 (MSE=0.3963, ASR=0.4159) Best: 0.4233
Epoch   60 | Train: 0.4712 | Val: 0.4114 (MSE=0.3656, ASR=0.4585) Best: 0.3977
Epoch   70 | Train: 0.4624 | Val: 0.4062 (MSE=0.3587, ASR=0.4750) Best: 0.3957
Epoch   80 | Train: 0.4522 | Val: 0.3962 (MSE=0.3500, ASR=0.4616) Best: 0.3910
Epoch   90 | Train: 0.4394 | Val: 0.3854 (MSE=0.3405, ASR=0.4489) Best: 0.3854
Epoch  100 | Train: 0.4331 | Val: 0.3837 (MSE=0.3378, ASR=0.4586) Best: 0.3768
Epoch  110 | Train: 0.4328 | Val: 0.3867 (MSE=0.3430, ASR=0.4372) Best: 0.3758
Epoch  120 | Train: 0.4258 | Val: 0.3842 (MSE=0.3402

## ارزیابی نهایی روی Test (همان روش notebook 15: فقط بلوک مرجع پیش‌بینی، بقیه از ground truth قرض گرفته می‌شود)

این یک مقایسه‌ی دقیقاً منصفانه با MAE=0.909 نوت‌بوک 15 است (همان روش ارزیابی، همان Test
split دست‌نخورده).

In [17]:
model.eval()
all_freq_mae = []

with torch.no_grad():
    for i in tqdm(test_idx, desc="Final evaluation on untouched test split"):
        s = raw_samples[i]
        bond_batch = Batch.from_data_list([bond_graphs_all[i]]).to(device)
        atom_batch = Batch.from_data_list([atom_graphs_all[i]]).to(device)

        ifc_pred_block = model(bond_batch, atom_batch)[0].cpu().numpy()  # (8,8,3,3)

        try:
            phonon = s['phonon_obj']
            full_ifc = s['fc_full'].copy()
            s2u_map = phonon.supercell.s2u_map
            unitcell_sc_indices = np.unique(s2u_map)
            # replace the reference block with the prediction, keep the rest as ground truth
            # (matching notebook 15's v4 evaluation protocol exactly)
            full_ifc[:, unitcell_sc_indices, :, :] = ifc_pred_block
            phonon.force_constants = full_ifc
            phonon.run_qpoints([[0, 0, 0]])
            pred_freqs = phonon.get_qpoints_dict()['frequencies'][0]

            true_peak = float(np.max(s['y_phonon']))
            pred_peak = float(np.max(np.abs(pred_freqs)))
            all_freq_mae.append(abs(true_peak - pred_peak))
        except Exception as e:
            continue

freq_mae = np.mean(all_freq_mae) if all_freq_mae else float('nan')
print(f"\nFrequency MAE (v4 architecture + symmetry augmentation, THz): {freq_mae:.4f}")
print(f"   v4 baseline (notebook 15, no augmentation):  0.909")
print(f"   This notebook (v4 + symmetry augmentation):  {freq_mae:.4f}")
improvement = (0.909 - freq_mae) / 0.909 * 100
print(f"   Improvement: {improvement:.1f}%" if not np.isnan(freq_mae) else "")

Final evaluation on untouched test split:   0%|          | 0/37 [00:00<?, ?it/s]

/tmp/ipykernel_58/697031601.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  pred_freqs = phonon.get_qpoints_dict()['frequencies'][0]



Frequency MAE (v4 architecture + symmetry augmentation, THz): 1.1934
   v4 baseline (notebook 15, no augmentation):  0.909
   This notebook (v4 + symmetry augmentation):  1.1934
   Improvement: -31.3%


## جمع‌بندی

این نوت‌بوک augmentation فیزیکی تأیید‌شده (notebook 19) را روی کل بخش train دیتاست (۲۸۶
ماده‌ی اصلی، بدون لمس val/test) اعمال کرد و معماری v4 (notebook 15، بدون هیچ تغییر
معماری) را روی داده‌ی گسترش‌یافته بازآموزی داد.

اگر MAE این نسخه پایین‌تر از baseline (0.909) باشد، فرضیه‌ی "augmentation داده، نه تغییر
معماری، راه درست بهبود است" تأیید می‌شود.

### ثبت در Obsidian
نتیجه‌ی این Notebook (MAE نهایی + تعداد نمونه‌ی augment‌شده) باید در
`08 - نقشه راه مقاله.md` به‌عنوان نتیجه‌ی فاز ۳ پروژه ثبت شود.